<a href="https://colab.research.google.com/github/zawuya/ZoeaLab/blob/master/CheckRobottxt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import json
import hashlib
import requests
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser
from bs4 import BeautifulSoup
from datetime import datetime

# ==== CONFIG ====
OUTPUT_DIR = "/content/drive/MyDrive/FM_Corpus/raw_data/shards"
MAX_SHARD_SIZE_MB = 100
URLS = [
    "https://www.bbc.com/swahili",
    "https://swahilivilla.blogspot.com/",
    # add more URLs here...
]
# ================

os.makedirs(OUTPUT_DIR, exist_ok=True)

def allowed_by_robots(url, user_agent="*"):
    """Check robots.txt compliance."""
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = RobotFileParser()
    try:
        rp.set_url(robots_url)
        rp.read()
        return rp.can_fetch(user_agent, url)
    except:
        return False

def extract_license(soup, url):
    """Try to extract license metadata from page head."""
    # Common license patterns (CC, copyrights, etc.)
    license_meta = soup.find("meta", {"name": "license"}) or \
                   soup.find("a", href=re.compile("creativecommons|license", re.I))
    if license_meta:
        return license_meta.get("content") if license_meta.has_attr("content") else license_meta.get("href")
    return "unknown"

def fetch_and_parse(url):
    """Fetch page, extract clean text + license."""
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        # Extract visible text
        text = " ".join([p.get_text() for p in soup.find_all("p")])
        text = re.sub(r"\s+", " ", text).strip()

        license_info = extract_license(soup, url)

        return {
            "url": url,
            "timestamp": datetime.utcnow().isoformat(),
            "license": license_info,
            "text": text
        }
    except Exception as e:
        print(f"[ERROR] Failed {url}: {e}")
        return None

def checksum(path):
    """SHA256 checksum of a file."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            h.update(chunk)
    return h.hexdigest()

# === Sharding logic ===
shard_index = 0
current_size = 0
current_shard = None
manifest = []

def new_shard():
    global shard_index, current_size, current_shard, manifest
    if current_shard:
        # finalize old shard with manifest
        shard_path = os.path.join(OUTPUT_DIR, f"shard_{shard_index:05d}.jsonl")
        mf = {
            "shard": shard_path,
            "num_records": len(manifest),
            "size_bytes": os.path.getsize(shard_path),
            "checksum": checksum(shard_path),
            "urls": [m["url"] for m in manifest],
            "created": datetime.utcnow().isoformat(),
        }
        with open(shard_path.replace(".jsonl", ".manifest.json"), "w") as f:
            json.dump(mf, f, indent=2)

    # start new shard
    shard_index += 1
    current_size = 0
    current_shard = open(os.path.join(OUTPUT_DIR, f"shard_{shard_index:05d}.jsonl"), "w", encoding="utf-8")
    manifest = []

def write_record(record):
    global current_size, current_shard, manifest
    if current_shard is None:
        new_shard()
    line = json.dumps(record, ensure_ascii=False) + "\n"
    current_shard.write(line)
    current_size += len(line.encode("utf-8"))
    manifest.append(record)
    if current_size > MAX_SHARD_SIZE_MB * 1024 * 1024:
        current_shard.close()
        new_shard()

# === Batch processing ===
for url in URLS:
    if allowed_by_robots(url):
        rec = fetch_and_parse(url)
        if rec and rec["text"]:
            write_record(rec)
    else:
        print(f"[SKIPPED robots.txt] {url}")

# Finalize last shard
if current_shard:
    current_shard.close()
    new_shard()

print("✅ Done. Shards + manifests saved in:", OUTPUT_DIR)


In [ ]:
# ==== 1️⃣ Mount Drive & prepare folder ====
from google.colab import drive
import os

# Mount Drive with force_remount to avoid errors
drive.mount('/content/drive', force_remount=True)

# Folder setup
OUTPUT_DIR = '/content/drive/MyDrive/FM_Corpus/raw_data/shards'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Output folder ready at:", OUTPUT_DIR)

# ==== 2️⃣ Imports ====
import re
import json
import hashlib
import requests
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser
from bs4 import BeautifulSoup
from datetime import datetime

# ==== 3️⃣ Config ====
MAX_SHARD_SIZE_MB = 100
URLS = [
    "https://www.bbc.com/swahili",
    "https://swahilivilla.blogspot.com/",
    # add more URLs here
]

# ==== 4️⃣ Functions ====
def allowed_by_robots(url, user_agent="*"):
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = RobotFileParser()
    try:
        rp.set_url(robots_url)
        rp.read()
        return rp.can_fetch(user_agent, url)
    except:
        return False

def extract_license(soup):
    license_meta = soup.find("meta", {"name": "license"}) or \
                   soup.find("a", href=re.compile("creativecommons|license", re.I))
    if license_meta:
        return license_meta.get("content") if license_meta.has_attr("content") else license_meta.get("href")
    return "unknown"

def fetch_and_parse(url):
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        text = " ".join([p.get_text() for p in soup.find_all("p")])
        text = re.sub(r"\s+", " ", text).strip()
        license_info = extract_license(soup)
        return {
            "url": url,
            "timestamp": datetime.utcnow().isoformat(),
            "license": license_info,
            "text": text
        }
    except Exception as e:
        print(f"[ERROR] Failed {url}: {e}")
        return None

def checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            h.update(chunk)
    return h.hexdigest()

# ==== 5️⃣ Sharding Logic ====
shard_index = 0
current_size = 0
current_shard = None
manifest = []

def new_shard():
    global shard_index, current_size, current_shard, manifest
    if current_shard:
        shard_path = os.path.join(OUTPUT_DIR, f"shard_{shard_index:05d}.jsonl")
        mf = {
            "shard": shard_path,
            "num_records": len(manifest),
            "size_bytes": os.path.getsize(shard_path),
            "checksum": checksum(shard_path),
            "urls": [m["url"] for m in manifest],
            "created": datetime.utcnow().isoformat(),
        }
        with open(shard_path.replace(".jsonl", ".manifest.json"), "w") as f:
            json.dump(mf, f, indent=2)

    shard_index += 1
    current_size = 0
    current_shard = open(os.path.join(OUTPUT_DIR, f"shard_{shard_index:05d}.jsonl"), "w", encoding="utf-8")
    manifest.clear()

def write_record(record):
    global current_size, current_shard, manifest
    if current_shard is None:
        new_shard()
    line = json.dumps(record, ensure_ascii=False) + "\n"
    current_shard.write(line)
    current_size += len(line.encode("utf-8"))
    manifest.append(record)
    if current_size > MAX_SHARD_SIZE_MB * 1024 * 1024:
        current_shard.close()
        new_shard()

# ==== 6️⃣ Batch Processing ====
for url in URLS:
    if allowed_by_robots(url):
        rec = fetch_and_parse(url)
        if rec and rec["text"]:
            write_record(rec)
    else:
        print(f"[SKIPPED robots.txt] {url}")

# Finalize last shard
if current_shard:
    current_shard.close()
    new_shard()

print("✅ Done. Shards + manifests saved in:", OUTPUT_DIR)
